In [3]:
# Imports
import pandas as pd
import numpy as np
from tabulate import tabulate
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, BayesianRidge

import warnings
warnings.filterwarnings('ignore')

## Utility Functions

In [4]:
def load_node_profile(node):
    filepath = f'../../tool/infrastructure-profiler/profiles/{node}.csv'

    with open(filepath) as f:
        data = f.readlines()[1].split(',')

    return {
        'ram': int(data[1].strip()),
        'cores': int(data[2].strip()),
        'io_read': float(data[3].strip()),
        'io_write': float(data[4].strip()),
        'sysbench': float(data[5].strip()),
        'z7b': float(data[6].strip()),
        'bench_time': float(data[7].strip())
    }

def combine_df(input_df, trace_df):
    rows = []

    for name, group in input_df.groupby('name'):
        no_files = str(len(group))
        filepaths = group['path']
        paths = ','.join([filepath.split('/')[-1].split('.')[-1] for filepath in filepaths])
        total_size = group['size'].sum() / 1073741824  # convert from B to GiB

        if len(trace_df[trace_df['name'] == name]) > 0:
            task_record = trace_df[trace_df['name'] == name].iloc[0]
            row = [task_record['task_id'], task_record['name'], task_record['%cpu'], 
                task_record['start'], task_record['complete'], task_record['complete'] - task_record['start'], 
                no_files, paths, total_size, task_record['name'].split(':')[-1].split(' ')[0], task_record['hostname']]
            rows.append(row)
        else:
            print('err:', name)

    combined = pd.DataFrame(rows, columns=['id', 'name', '%cpu', 'start', 'complete', 'runtime', 'no_files', 'files', 'input_gib', 'short_name', 'hostname'])
    return combined

def load_profiling_runs(node, workflows, samples, node_profiles):
    node_profile = node_profiles[node]
    runs = {}

    for workflow, samples in zip(workflows, samples):
        runs[workflow] = {}

        for sample in samples:
            sample_dir = f'{node}/{workflow}/{sample}/'
            input_df = pd.read_csv(sample_dir + 'input.csv', sep=';')
            trace_df = pd.read_csv(sample_dir + 'trace.csv')
            runs[workflow][sample] = combine_df(input_df, trace_df)

    return (node_profile, runs)

def load_workflow_execution(cluster, workflow, iteration):
    execution_dir = f'../{cluster}/{workflow}/{iteration}/'
    input_df = pd.read_csv(execution_dir + 'input.csv', sep=';')
    trace_df = pd.read_csv(execution_dir + 'trace.csv')
    return combine_df(input_df, trace_df)

def load_ichnos_trace(cluster, workflow, iteration, gov):
    short_cluster = cluster.split('-')[0]
    trace_loc = f'../{cluster}/{workflow}/ichnos/{short_cluster}-{workflow}-{iteration}-1-{gov}-trace.csv'
    return pd.read_csv(trace_loc)

def process_ichnos_summary(raw):
    core_total_energy = float(raw[7].split(': ')[-1][:-3])
    mem_dynamic_energy = float(raw[9].split(': ')[-1][:-3])
    mem_static_energy = float(raw[15].split(': ')[-1][:-3])
    return {
        'core_total': core_total_energy, 
        'mem_dynamic': mem_dynamic_energy, 
        'mem_static': mem_static_energy, 
        'total': core_total_energy + mem_dynamic_energy + mem_static_energy
    } 

def load_ichnos_summary(cluster, workflow, iteration, gov):
    short_cluster = cluster.split('-')[0]
    with open(f'../{cluster}/{workflow}/ichnos/{short_cluster}-{workflow}-{iteration}-1-{gov}-summary.txt', 'r') as file:
        raw = file.readlines()
    raw = [line.strip() for line in raw]
    return process_ichnos_summary(raw)

def load_rapl_summary(cluster, workflow, iteration):
    rapl_vals = pd.read_csv(f'../{cluster}/{workflow}-runs.csv')
    return rapl_vals.iloc[[iteration - 1]]

def load_power_model(node, setting=''):
    if setting != '':
        filename = f'{setting}-model.txt'
    else: 
        filename = 'model.txt'

    with open(f'../../tool/infrastructure-profiler/raw/{node}/{filename}', 'r') as f:
        data = [line.strip() for line in f.readlines()[1:]]

    coeff = float(data[0].split(',')[0][2:].strip())
    inter = float(data[0].split(',')[1][:-2].strip())
    draw = float(data[1].split(':')[1].strip())

    if draw == 0:
        draw = 0.392

    return (coeff, inter, draw)

def get_task_data_from_experiment(task_name, experiment):
    task_data = experiment[experiment['short_name'] == task_name]
    key_data = task_data[['input_gib', 'runtime', '%cpu']]
    key_data.columns = ['data', 'time', 'usage']
    key_data.reset_index().drop('index', axis=1)
    return key_data

In [5]:
class MedianModel:
    def __init__(self, median):
        self.median = median
    
    def predict(self, x):
        x = np.asarray(x)
        value = x.shape[0]
        return np.full(value, self.median)

## Setup Data

In [6]:
# Constants
HU = 'hu'
HU_40 = 'hu-c40'
HU_42 = 'hu-c42'
HU_44 = 'hu-c44'
HU_45 = 'hu-c45'
GPG = 'gpg'
GPG_16 = 'gpgnode-16'
GPG_22 = 'gpgnode-22'
TU = 'tu'
TU_06 = 'tu-06'
AWS = 'aws'
AWS_c5n = 'aws-metal-c5n'
AWS_z1d = 'aws-metal-z1d'
wfs = {
    GPG: ['rnaseq', 'chipseq'], 
    HU: ['rnaseq', 'chipseq', 'rangeland', 'sarek']
}
VR = 'VR'
RF = 'RandomForest'
LR = 'LinearRegression'
BR = 'BayesianRidge'

In [7]:
# Node Profiles + Power Models (todo)
nodes = [AWS_c5n, AWS_z1d, GPG_16, GPG_22, TU_06, HU_40, HU_42] #, HU_44, HU_45]
node_profiles = {}

for node in nodes:
    profile = load_node_profile(node)
    node_profiles[node] = profile

In [8]:
# Workflow Executions 
wf_executions = {}

for cluster, cluster_path in zip([GPG, HU], ['gu-cluster', 'hu-cluster']):
    wf_executions[cluster] = {}
    for wf in wfs[cluster]:
        wf_executions[cluster][wf] = {}
        for iteration in range(1, 4):
            wf_executions[cluster][wf][iteration] = {}
            wf_executions[cluster][wf][iteration]['run'] = load_workflow_execution(cluster_path, wf, iteration)

err: NFCORE_SAREK:SAREK:BAM_VARIANT_CALLING_TUMOR_ONLY_ALL:BAM_VARIANT_CALLING_SINGLE_STRELKA:MERGE_STRELKA (SRR11452475)


In [9]:
# Workflow RAPL Total
for cluster, cluster_path in zip([GPG, HU], ['gu-cluster', 'hu-cluster']):
    for wf in wfs[cluster]:
        for iteration in range(1, 4):
            wf_executions[cluster][wf][iteration]['rapl'] = load_rapl_summary(cluster_path, wf, iteration)

In [10]:
# Workflow Ichnos Total
for cluster, cluster_path, gov in zip([GPG, HU], ['gu-cluster', 'hu-cluster'], ['ondemand_linear', 'schedutil_linear']):
    for wf in wfs[cluster]:
        for iteration in range(1, 4):
            wf_executions[cluster][wf][iteration]['ichnos'] = load_ichnos_summary(cluster_path, wf, iteration, gov)

In [11]:
# Join Workflow Traces with Ichnos Traces 
def add_energy_with_task_id(run, ichnos):
    combined = pd.merge(run, ichnos[['id', 'core_kwh', 'mem_kwh']], on='id').drop_duplicates()
    return combined[['id', 'name', '%cpu', 'start', 'complete', 'runtime', 'no_files', 'files', 'input_gib', 'short_name', 'hostname', 'core_kwh', 'mem_kwh']]

# Workflow Ichnos Total
for cluster, cluster_path, gov in zip([GPG, HU], ['gu-cluster', 'hu-cluster'], ['ondemand_linear', 'schedutil_linear']):
    for wf in wfs[cluster]:
        for iteration in range(1, 4):
            run = wf_executions[cluster][wf][iteration]['run']
            ichnos_trace = load_ichnos_trace(cluster_path, wf, iteration, gov)
            df_with_energy = add_energy_with_task_id(run, ichnos_trace)
            wf_executions[cluster][wf][iteration]['run'] = df_with_energy

In [12]:
# Compare RAPL Total to Ichnos Total
for cluster, cluster_path, socket in zip([GPG, HU], ['gu-cluster', 'hu-cluster'], ['_2', '']):
    print('===', cluster, '===')
    for wf in wfs[cluster]:
        for iteration in range(1, 4):
            ichnos_total = wf_executions[cluster][wf][iteration]['ichnos']['total']
            ichnos_core = wf_executions[cluster][wf][iteration]['ichnos']['core_total']
            ichnos_mem = wf_executions[cluster][wf][iteration]['ichnos']['mem_dynamic'] + wf_executions[cluster][wf][iteration]['ichnos']['mem_static']

            rapl_core = wf_executions[cluster][wf][iteration]['rapl'][f'active_pkg{socket}'].iloc[0]

            if wf_executions[cluster][wf][iteration]['rapl'][f'idle_dram{socket}'].iloc[0] > wf_executions[cluster][wf][iteration]['rapl'][f'dram{socket}'].iloc[0]:
                rapl_mem = wf_executions[cluster][wf][iteration]['rapl'][f'dram{socket}'].iloc[0]
            else:
                rapl_mem = wf_executions[cluster][wf][iteration]['rapl'][f'active_dram{socket}'].iloc[0]

            rapl_total = rapl_core + rapl_mem
            wf_executions[cluster][wf][iteration]['rapl']['core'] = rapl_core
            wf_executions[cluster][wf][iteration]['rapl']['mem'] = rapl_mem
            core_error = abs(rapl_core - ichnos_core) / rapl_core * 100
            mem_error = abs(rapl_mem - ichnos_mem) / rapl_mem * 100
            total_error = abs(rapl_total - ichnos_total) / rapl_total * 100

            print(f'{wf}-{iteration} ichnos: {ichnos_total:.2f}kWh rapl: {rapl_total:.2f}kWh error {total_error:.2f}%')

=== gpg ===
rnaseq-1 ichnos: 1.51kWh rapl: 1.41kWh error 7.11%
rnaseq-2 ichnos: 1.53kWh rapl: 1.45kWh error 6.00%
rnaseq-3 ichnos: 1.53kWh rapl: 1.42kWh error 7.87%
chipseq-1 ichnos: 2.00kWh rapl: 1.91kWh error 5.03%
chipseq-2 ichnos: 1.99kWh rapl: 1.84kWh error 7.77%
chipseq-3 ichnos: 2.02kWh rapl: 1.94kWh error 4.22%
=== hu ===
rnaseq-1 ichnos: 2.26kWh rapl: 2.46kWh error 8.08%
rnaseq-2 ichnos: 2.27kWh rapl: 2.52kWh error 9.85%
rnaseq-3 ichnos: 2.29kWh rapl: 2.62kWh error 12.53%
chipseq-1 ichnos: 4.69kWh rapl: 5.12kWh error 8.29%
chipseq-2 ichnos: 4.68kWh rapl: 5.16kWh error 9.36%
chipseq-3 ichnos: 4.67kWh rapl: 5.19kWh error 9.99%
rangeland-1 ichnos: 2.16kWh rapl: 2.48kWh error 12.75%
rangeland-2 ichnos: 2.17kWh rapl: 2.49kWh error 13.12%
rangeland-3 ichnos: 2.16kWh rapl: 2.41kWh error 10.39%
sarek-1 ichnos: 4.86kWh rapl: 5.19kWh error 6.49%
sarek-2 ichnos: 4.81kWh rapl: 5.32kWh error 9.48%
sarek-3 ichnos: 4.81kWh rapl: 5.32kWh error 9.57%


In [13]:
# Load workflow profiling runs
wf_profiles = {}

# TU_06
wfs = ['rnaseq', 'chipseq']
samples = [['sample_b', 'sample_c', 'sample_d'], ['sample_e', 'sample_f', 'sample_g', 'sample_h']]
profile, runs = load_profiling_runs(TU_06, wfs, samples, node_profiles)
wf_profiles[TU_06] = {
    'node_profile': profile,
    'executions': runs,
    'power_model': load_power_model(TU_06)
}

# HU_40
wfs = ['rnaseq', 'chipseq']
samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d', 'sample_e'], ['sample_f', 'sample_g', 'sample_h', 'sample_i', 'sample_j']]
profile, runs = load_profiling_runs(HU_40, wfs, samples, node_profiles)
wf_profiles[HU_40] = {
    'node_profile': profile,
    'executions': runs,
    'power_model': load_power_model(HU_40, 'schedutil')
}

# HU_42
wfs = ['rnaseq', 'chipseq']
samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d', 'sample_e'], ['sample_g', 'sample_h', 'sample_i', 'sample_j']]  # sample_f to be added back
profile, runs = load_profiling_runs(HU_42, wfs, samples, node_profiles)
wf_profiles[HU_42] = {
    'node_profile': profile,
    'executions': runs,
    'power_model': load_power_model(HU_42, 'schedutil')
}

# HU_44
# wfs = ['rnaseq', 'chipseq']
# samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d', 'sample_e'], ['sample_f', 'sample_g', 'sample_h', 'sample_i', 'sample_j']]  
# profile, runs = load_profiling_runs(HU_44, wfs, samples, node_profiles)
# wf_profiles[HU_44] = {
#     'node_profile': profile,
#     'executions': runs,
#     'power_model': load_power_model(HU_44, 'schedutil')
# }

# HU_45
# wfs = ['rnaseq', 'chipseq']
# samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d', 'sample_e'], ['sample_f', 'sample_g', 'sample_h', 'sample_i', 'sample_j']]  
# profile, runs = load_profiling_runs(HU_45, wfs, samples, node_profiles)
# wf_profiles[HU_45] = {
#     'node_profile': profile,
#     'executions': runs,
#     'power_model': load_power_model(HU_45, 'schedutil')
# }

# gpgnode-16
wf_profiles[GPG_16] = {
    'node_profile': node_profiles[GPG_16],
    'power_model': load_power_model(GPG_16, 'ondemand')
}

## Evaluation

In [ ]:
# Prediction and Verification
def compute_active_time_per_host(execution):
    tasks_by_host = {}

    for _, record in execution.iterrows():
        host = record['hostname']
        start_ts = record['start']
        end_ts = record['complete']

        if host in tasks_by_host:
            tasks_by_host[host].append((start_ts, end_ts))
        else:
            tasks_by_host[host] = [(start_ts, end_ts)]

    active_time_by_host = {}

    for host, intervals in tasks_by_host.items():
        if not intervals:
            continue

        intervals.sort(key=lambda x: x[0])  # sort by start timestamp
        merged = []
        current_start, current_end = intervals[0]

        for start, end in intervals[1:]:
            if start <= current_end:  # overlap or contiguous
                current_end = max(current_end, end)
            else:
                merged.append((current_start, current_end))
                current_start, current_end = start, end
        merged.append((current_start, current_end))

        total_ms: float = sum(end - start for start, end in merged)
        active_time_by_host[host] = total_ms / 1000 / 3600  # return time in hours

    return active_time_by_host

# @Vasilis - where we take profiling runs -> to make task predictions models and persist cpu usage
def create_prediction_model(runs, model_type):
    models = {}

    for workflow in runs.keys():
        models[workflow] = {}
        samples = list(runs[workflow].keys()) # ['sample_a', ...]
        tasks = runs[workflow][samples[0]]['short_name'].unique() # potentially where missing tasks come from? 

        for task in tasks:
            task_data = pd.DataFrame(columns=['data', 'time', 'usage'], dtype='float64')
            for sample in samples: 
                experiment_data = get_task_data_from_experiment(task, runs[workflow][sample])
                task_data = pd.concat([task_data, experiment_data])

            # based off lotaru (what if we have some correlation but not a strong one?)
            # we can also use io_read as a proxy for full input size per task
            pearson = np.corrcoef(task_data['data'].to_numpy(), task_data['time'].to_numpy())[0, 1]
            models[workflow][task] = {}
            models[workflow][task]['usage'] = np.mean(task_data['usage'])

            if np.isnan(pearson) or pearson < 0.75:
               models[workflow][task]['model'] = MedianModel(np.median(task_data['time'].to_numpy())) 
               # area to improve upon what Lotaru does
               # @Vasilis - can check % of how many tasks scale linearly / not 
            else:
                if model_type == VR:
                    model = SVR()
                elif model_type == RF:
                    model = RandomForestRegressor()
                elif model_type == BR:
                    model = BayesianRidge(fit_intercept=True)
                else:
                    model = LinearRegression()
                # todo read up more on these regression models

                models[workflow][task]['model'] = model.fit(task_data['data'].values.reshape(-1, 1), task_data['time'].values) 

    return models

def get_translation_factor(local_profile, target_profile):
    local_write_val = local_profile['io_write'] / local_profile['ram']
    local_read_val = local_profile['io_read'] / local_profile['ram']
    target_write_val = target_profile['io_write'] / target_profile['ram']
    target_read_val = target_profile['io_read'] / target_profile['ram']
    return (0.25 * local_profile['z7b'] / target_profile['z7b']) + \
    (0.25 * local_profile['sysbench'] / target_profile['sysbench']) + \
    (0.25 * local_write_val / target_write_val) + \
    (0.25 * local_read_val / target_read_val)
    #(0.25 * local_profile['bench_time'] / target_profile['bench_time']) + \

# ^ io + cpu 0.5 @Vasilis this is the function for translation factor, Lotaru does -- 
# ft = 0.5 * (cpu local / cpu_target) + 0.5 * (io local / io target)

def get_error(predicted, actual):
    return abs(actual - predicted) / actual * 100

def reflector(cluster, task_name):
    return []

def predict_wf(wf_model, workflow, factor, local_profile, target_profile, target_power_model):
    overall_error = []
    energy_error = []
    missing_tasks = []    
    headers = ['task', 'input (GiB)', 'pred_time (s)', 'actual_time (s)', 'error (%)', 'pred_usage (%)', 'actual_usage (%)', 'pred_cpu (kWh)', 'actual_cpu (kWh)', 'pred_mem (kWh)', 'actual_mem (kWh)', 'node']
    data = []
    actual_execution = workflow['run']
    total_cpu_dynamic_energy = 0.0
    total_mem_dynamic_energy = 0.0

    for _, record in actual_execution.iterrows():
        task_name = record['short_name']
        data_size = record['input_gib']
        actual_runtime = record['runtime']

        # exclude tasks that run for 0-5 seconds
        if actual_runtime > 5000:
            actual_usage = record["%cpu"]
            node = record['hostname']

            if task_name in wf_model.keys():
                if local_profile['cores'] != target_profile['cores']:
                    predicted_runtime = wf_model[task_name]['model'].predict([[data_size]])[0] * factor * (wf_model[task_name]['usage'] / target_profile['cores'])
                    pred_usage = (pred_usage / local_profile['cores']) * target_profile['cores']
                else:
                    predicted_runtime = wf_model[task_name]['model'].predict([[data_size]])[0] * factor  

                pred_usage = wf_model[task_name]['usage']

                if predicted_runtime > 0:
                    error = (abs(predicted_runtime - actual_runtime) / actual_runtime) * 100 
                    overall_error.append(error)
            else:
                missing_tasks.append(task_name) 
                predicted_runtime = 0
                error = 100
                pred_usage = 0
        
            # estimate energy consumption (ichnos logic)
            time_h = predicted_runtime / 1000 / 3600  # convert from ms to hours
            cpu_usage = pred_usage / target_profile['cores']
            memory = data_size
            core_consumption = time_h * (target_power_model[0] * cpu_usage) * 0.001
            mem_consumption = memory * target_power_model[2] * time_h * 0.001

            total_cpu_dynamic_energy += core_consumption
            total_mem_dynamic_energy += mem_consumption

            predicted_energy = core_consumption + mem_consumption
            actual_energy = record['core_kwh'] + record['mem_kwh']

            if actual_energy > 0 and predicted_energy > 0:
                energy_error.append(get_error(predicted_energy, actual_energy))

            # report
            data.append([task_name, f'{data_size:.2f}', f'{predicted_runtime / 1000:.2f}', f'{actual_runtime / 1000:.2f}', error, f'{pred_usage:.2f}', f'{actual_usage}', f'{core_consumption}', f'{record['core_kwh']}', f'{mem_consumption}', f'{record['mem_kwh']}', node])

    print(tabulate(data, headers, tablefmt='orgtbl'))
    print('missing tasks:', missing_tasks)
    print('avg. runtime prediction error:', np.mean(overall_error))  
    print('avg. energy prediction error:', np.mean(energy_error))

    # static power consumption
    total_cpu_static_energy = 0.0
    total_mem_static_energy = 0.0
    active_time_per_host = compute_active_time_per_host(actual_execution)
    for host in active_time_per_host.keys():
        total_cpu_static_energy += active_time_per_host[host] * target_power_model[1] * 0.001  # convert to kWh
        total_mem_static_energy += active_time_per_host[host] * target_power_model[2] * target_profile['ram'] * 0.001 # convert to kWh

    print('augur dynamic task energy: cpu', total_cpu_dynamic_energy, 'mem', total_mem_dynamic_energy)
    print('augur static energy: cpu', total_cpu_static_energy, 'mem', total_mem_static_energy)
    augur_total_cpu = total_cpu_dynamic_energy + total_cpu_static_energy
    augur_total_mem = total_mem_dynamic_energy+total_mem_static_energy
    augur_total = augur_total_cpu + augur_total_mem
    ichnos_total = workflow['ichnos']['total']
    rapl_total = float(workflow['rapl']['core']) + float(workflow['rapl']['mem'])
    print('augur total cpu: ', augur_total_cpu, 'mem', augur_total_mem, 'total', augur_total)
    print('ichnos:', ichnos_total, get_error(augur_total, ichnos_total))
    print('rapl:', rapl_total, 'error:', get_error(augur_total, rapl_total))

In [15]:
# @Vasilis
print(wf_profiles[HU_40]['executions']['rnaseq']['sample_a'])

    id                                               name   %cpu  \
0    2  NFCORE_RNASEQ:PREPARE_GENOME:CUSTOM_GETCHROMSI...   99.9   
1    1  NFCORE_RNASEQ:PREPARE_GENOME:GTF_FILTER (genom...   99.8   
2    6  NFCORE_RNASEQ:PREPARE_GENOME:MAKE_TRANSCRIPTS_...  100.1   
3    7  NFCORE_RNASEQ:PREPARE_GENOME:SALMON_INDEX (gen...  425.5   
4   16  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS...  135.8   
5   14  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS...  123.1   
6   15  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS...  120.3   
7   12  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS...  142.2   
8   10  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS...  563.6   
9    9  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:STAR_ALIGN_IGE...  274.3   
10  34  NFCORE_RNASEQ:RNASEQ:BAM_MARKDUPLICATES_PICARD...  132.2   
11  32  NFCORE_RNASEQ:RNASEQ:BAM_MARKDUPLICATES_PICARD...   95.0   
12  33  NFCORE_RNASEQ:RNASEQ:BAM_MARKDUPLICATES_PICARD...  119.6   
13  13  NFCORE_RNASEQ:RNASEQ:BAM_MARKDUPLICATES_

In [37]:
# TEST: hu_c40 to hu-cluster execution
hu_wf_models = create_prediction_model(wf_profiles[HU_40]['executions'], LR)
local_profile = wf_profiles[HU_40]['node_profile']
target_profile = wf_profiles[HU_40]['node_profile']
factor = get_translation_factor(local_profile, target_profile)
target_power_model = wf_profiles[HU_40]['power_model']
hu_rnaseq_model = hu_wf_models['rnaseq']
predict_wf(hu_rnaseq_model, wf_executions[HU]['rnaseq'][1], factor, local_profile, target_profile, target_power_model)

| task                         |   input (GiB) |   pred_time (s) |   actual_time (s) |   error (%) |   pred_usage (%) |   actual_usage (%) |   pred_cpu (kWh) |   actual_cpu (kWh) |   pred_mem (kWh) |   actual_mem (kWh) | node          |
|------------------------------+---------------+-----------------+-------------------+-------------+------------------+--------------------+------------------+--------------------+------------------+--------------------+---------------|
| SAMTOOLS_FLAGSTAT            |          1.58 |           13.74 |                14 |   1.82186   |           137.09 |              132.8 |      1.47798e-05 |        1.49387e-05 |      5.02712e-07 |        2.06612e-06 | hu-worker-c42 |
| SAMTOOLS_FLAGSTAT            |          1.58 |           13.74 |                14 |   1.82157   |           137.09 |              133.5 |      1.47799e-05 |        1.44533e-05 |      5.02715e-07 |        1.98576e-06 | hu-worker-c44 |
| SAMTOOLS_FLAGSTAT            |          1.58 |    

In [38]:
# TEST: hu_c40 model to gu-cluster execution
local_profile = wf_profiles[HU_40]['node_profile']
target_profile = wf_profiles[GPG_16]['node_profile']
factor = get_translation_factor(local_profile, target_profile)
target_power_model = wf_profiles[GPG_16]['power_model']
hu_rnaseq_model = hu_wf_models['rnaseq']
predict_wf(hu_rnaseq_model, wf_executions[GPG]['rnaseq'][1], factor, local_profile, target_profile, target_power_model)

| task                     |   input (GiB) |   pred_time (s) |   actual_time (s) |   error (%) |   pred_usage (%) |   actual_usage (%) |   pred_cpu (kWh) |   actual_cpu (kWh) |   pred_mem (kWh) |   actual_mem (kWh) | node       |
|--------------------------+---------------+-----------------+-------------------+-------------+------------------+--------------------+------------------+--------------------+------------------+--------------------+------------|
| CUSTOM_GETCHROMSIZES     |          2.93 |           23.17 |                19 |    21.9242  |           100.06 |               60.9 |      1.26891e-05 |        6.5271e-06  |      6.2194e-06  |        1.04912e-05 | gpgnode-18 |
| GTF_FILTER               |          3.78 |           42.07 |                30 |    40.2334  |            99.92 |               79.4 |      2.3012e-05  |        1.34367e-05 |      1.45824e-05 |        1.65651e-05 | gpgnode-18 |
| MAKE_TRANSCRIPTS_FASTA   |          3.71 |          120.13 |                96

In [39]:
# TEST: hu_c40 model to gu-cluster execution
local_profile = wf_profiles[HU_40]['node_profile']
target_profile = wf_profiles[GPG_16]['node_profile']
factor = get_translation_factor(local_profile, target_profile)
target_power_model = wf_profiles[GPG_16]['power_model']
hu_chipseq_model = hu_wf_models['chipseq']
predict_wf(hu_chipseq_model, wf_executions[GPG]['chipseq'][1], factor, local_profile, target_profile, target_power_model)

| task                                |   input (GiB) |   pred_time (s) |   actual_time (s) |   error (%) |   pred_usage (%) |   actual_usage (%) |   pred_cpu (kWh) |   actual_cpu (kWh) |   pred_mem (kWh) |   actual_mem (kWh) | node       |
|-------------------------------------+---------------+-----------------+-------------------+-------------+------------------+--------------------+------------------+--------------------+------------------+--------------------+------------|
| SAMTOOLS_FLAGSTAT                   |          1.03 |           19.28 |                15 |    28.511   |           136.95 |              116.8 |      1.44515e-05 |        1.09291e-05 |      1.81208e-06 |        1.78867e-05 | gpgnode-14 |
| SAMTOOLS_FLAGSTAT                   |          1.09 |           20.45 |                16 |    27.8405  |           136.95 |              116.2 |      1.53345e-05 |        1.00927e-05 |      2.05141e-06 |        1.77917e-05 | gpgnode-16 |
| SAMTOOLS_FLAGSTAT                 

In [40]:
# TEST: hu_c40 to hu-cluster execution
hu_wf_models = create_prediction_model(wf_profiles[HU_40]['executions'], LR)
local_profile = wf_profiles[HU_40]['node_profile']
target_profile = wf_profiles[HU_40]['node_profile']
factor = get_translation_factor(local_profile, target_profile)
target_power_model = wf_profiles[HU_40]['power_model']
hu_chipseq_model = hu_wf_models['chipseq']
predict_wf(hu_chipseq_model, wf_executions[HU]['chipseq'][1], factor, local_profile, target_profile, target_power_model)

| task                          |   input (GiB) |   pred_time (s) |   actual_time (s) |   error (%) |   pred_usage (%) |   actual_usage (%) |   pred_cpu (kWh) |   actual_cpu (kWh) |   pred_mem (kWh) |   actual_mem (kWh) | node          |
|-------------------------------+---------------+-----------------+-------------------+-------------+------------------+--------------------+------------------+--------------------+------------------+--------------------+---------------|
| SAMTOOLS_FLAGSTAT             |          0.89 |            6.28 |                 8 |  21.4525    |           136.95 |              144.9 |      6.74986e-06 |        8.87211e-06 |      1.30052e-07 |        2.40313e-06 | hu-worker-c45 |
| SAMTOOLS_FLAGSTAT             |          0.82 |            5.78 |                 7 |  17.387     |           136.95 |              147.3 |      6.21182e-06 |        8.08764e-06 |      1.09109e-07 |        1.94399e-06 | hu-worker-c40 |
| SAMTOOLS_FLAGSTAT             |          1.29 